In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('transaction.csv')

In [ ]:
df.columns.tolist()

In [ ]:
df.head(10)

In [ ]:
print(df.isnull().sum().sort_values(ascending=False)) 

In [ ]:
for c1, c2 in df.dtypes.reset_index().values:
    if c2=='O':
        print(df[c1].value_counts())
        print(f'\n > **Total Null Values:** {df[c1].isnull().sum()}')
        print(f'\n > **Total Unique Values:** {len(df[c1].unique())}')
        print('-' * 20 + '\n')

In [ ]:
# One-Hot Encode 'type'
type_dummies = pd.get_dummies(df['type'], prefix='type')
df = pd.concat([df, type_dummies], axis=1)
df.drop('type', axis=1, inplace=True)


In [ ]:
df.head()

In [ ]:
# Frequency Encoding for 'nameOrig'
freq_encoding_orig = df['nameOrig'].value_counts() / len(df)
df['nameOrig_freq'] = df['nameOrig'].map(freq_encoding_orig)

# Frequency Encoding for 'nameDest'
freq_encoding_dest = df['nameDest'].value_counts() / len(df)
df['nameDest_freq'] = df['nameDest'].map(freq_encoding_dest)

# Drop original columns
df.drop(['nameOrig', 'nameDest'], axis=1, inplace=True)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical features
numerical_features = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

# Initialize scaler
scaler = StandardScaler()

# Fit and transform
df[numerical_features] = scaler.fit_transform(df[numerical_features])


In [ ]:
# Balance differences for Origin
df['balance_diff_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']

# Balance differences for Destination
df['balance_diff_dest'] = df['oldbalanceDest'] - df['newbalanceDest']

# Optionally, drop original balance features if differences are more informative
df.drop(['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest'], axis=1, inplace=True)


In [ ]:
# 4. Feature Engineering: Time-Based Features
# Define constants
MINUTES_IN_HOUR = 60
MINUTES_IN_DAY = 24 * MINUTES_IN_HOUR

# Calculate minute within the day
df['minute_of_day'] = df['step'] % MINUTES_IN_DAY

# Extract hour and minute
df['hour'] = (df['minute_of_day'] // MINUTES_IN_HOUR).astype(int)
df['minute'] = (df['minute_of_day'] % MINUTES_IN_HOUR).astype(int)

# Create cyclical features
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)

# (Optional) Extract day features if applicable
DAYS_IN_WEEK = 7
df['day'] = (df['step'] // MINUTES_IN_DAY).astype(int)
df['day_of_week'] = df['day'] % DAYS_IN_WEEK
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / DAYS_IN_WEEK)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / DAYS_IN_WEEK)

# Drop intermediate time-related columns
df.drop(['minute_of_day', 'hour', 'minute', 'day', 'day_of_week'], axis=1, inplace=True, errors='ignore')

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X = df.drop('isFraud', axis=1)
y = df['isFraud']

# 8. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Data split into training and testing sets.")

# 9. Handle class imbalance with SMOTE (only on training data)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [ ]:
X_train_res

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

# After SMOTE, perform random undersampling
rus = RandomUnderSampler(sampling_strategy=1.0, random_state=42)  # Set desired ratio here
X_resampled, y_resampled = rus.fit_resample(X_train_res, y_train_res)
X_resampled

In [ ]:
import pandas as pd

# Assuming X_resampled and y_resampled are the features and target arrays
data_resampled = pd.concat([X_resampled, y_resampled], axis=1)

# Downsample to 200,000 rows
data_resampled_downsampled = data_resampled.sample(n=200000, random_state=42)
X_final = data_resampled_downsampled.drop('isFraud', axis=1)  # Replace 'target_column' with actual target column name
y_final = data_resampled_downsampled['isFraud']

In [ ]:
X_final

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for the plot
sns.set(style='whitegrid')
# Create a bar plot for the target variable 'isFraud'
plt.figure(figsize=(6, 4))
sns.countplot(x=y_final, palette='viridis')

# Add title and labels
plt.title('Distribution of Fraud and Non-Fraud Transactions')
plt.xlabel('Transaction Type (0: Non-Fraud, 1: Fraud)')
plt.ylabel('Count')

# Show plot
plt.show()
X_final

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lazypredict.Supervised import LazyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Assuming you have your data split into X and y
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define your selected models
selected_models = {
    'KNeighborsClassifier': KNeighborsClassifier(),
    'RandomForestClassifier': RandomForestClassifier(),
    'XGBClassifier': XGBClassifier(eval_metric='logloss'),  # Set eval_metric to avoid warning
    'LightGBMClassifier': LGBMClassifier(),
    'CatBoostClassifier': CatBoostClassifier(silent=True)  # Set silent=True to suppress warnings
}

# Store results
results = []

# Train and evaluate each selected model
for model_name, model in selected_models.items():
    # Fit the model
    model.fit(X_final, y_final)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate ROC AUC score
    roc_auc = roc_auc_score(y_test, y_pred)
    
    # Append results
    results.append({'Model': model_name, 'ROC AUC': roc_auc})

# Convert results to DataFrame for better visualization
import pandas as pd

results_df = pd.DataFrame(results)

# Display results sorted by ROC AUC
print(results_df.sort_values(by='ROC AUC', ascending=False))


In [ ]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}
clf = GridSearchCV(estimator=rf_classifier, 
                               param_grid=rf_param_grid, 
                               scoring='roc_auc', 
                               cv=5, 
                               verbose=1, 
                               n_jobs=-1)
clf.fit(X_final, y_final)

In [ ]:
print("Best parameters for RandomForestClassifier:", clf.best_params_)
print("Best ROC AUC score for RandomForestClassifier:", clf.best_score_)

In [ ]:
from sklearn.metrics import accuracy_score
print(accuracy_score(ypred,y_test))

In [ ]:
xgb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [5, 7, 10],
    'subsample': [0.8, 0.9, 1.0],
}

xgb_grid_search = GridSearchCV(estimator=xgb_classifier, 
                                param_grid=xgb_param_grid, 
                                scoring='roc_auc', 
                                cv=5, 
                                verbose=1, 
                                n_jobs=-1)

xgb_grid_search.fit(X_final, y_final)

# Best parameters and score for XGBClassifier
print("Best parameters for XGBClassifier:", xgb_grid_search.best_params_)
print("Best ROC AUC score for XGBClassifier:", xgb_grid_search.best_score_)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Best parameters from the grid search
best_params_rf = {
    'max_depth': None, 
    'min_samples_leaf': 1, 
    'min_samples_split': 2, 
    'n_estimators': 200
}

# Initialize the RandomForestClassifier with the best parameters
rf_model = RandomForestClassifier(
    max_depth=best_params_rf['max_depth'],
    min_samples_leaf=best_params_rf['min_samples_leaf'],
    min_samples_split=best_params_rf['min_samples_split'],
    n_estimators=best_params_rf['n_estimators'],
    random_state=42  # Set a random state for reproducibility
)

# Train the model with your training data
rf_model.fit(X_final, y_final)

# Optionally, make predictions and evaluate the model
y_pred = rf_model.predict(X_test)

# If you want to calculate the ROC AUC score for the test set
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])

print(f"ROC AUC score for RandomForestClassifier on test set: {roc_auc}")


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# Best parameters from the grid search
best_params_xgb = {
    'learning_rate': 0.1, 
    'max_depth': 10, 
    'n_estimators': 200, 
    'subsample': 0.9
}

# Initialize the XGBClassifier with the best parameters
xgb_model = XGBClassifier(
    learning_rate=best_params_xgb['learning_rate'],
    max_depth=best_params_xgb['max_depth'],
    n_estimators=best_params_xgb['n_estimators'],
    subsample=best_params_xgb['subsample'],
    use_label_encoder=False,   # Disable label encoding if you're using newer XGBoost versions
    random_state=42  # Set a random state for reproducibility
)

# Train the model with your training data
xgb_model.fit(X_final, y_final)

# Optionally, make predictions and evaluate the model
y_pred = xgb_model.predict(X_test)

# Calculate the ROC AUC score for the test set
roc_auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])

print(f"ROC AUC score for XGBClassifier on test set: {roc_auc}")


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
from xgboost import plot_importance

fig = plt.figure(figsize = (14, 9))
ax = fig.add_subplot(111)

colours = plt.cm.Set1(np.linspace(0, 1, 9))

ax = plot_importance(xgb_model, height = 1, color = colours, grid = False, \
                     show_values = False, importance_type = 'cover', ax = ax);
for axis in ['top','bottom','left','right']:
            ax.spines[axis].set_linewidth(2)
        
ax.set_xlabel('importance score', size = 16);
ax.set_ylabel('features', size = 16);
ax.set_yticklabels(ax.get_yticklabels(), size = 12);
ax.set_title('Ordering of features by importance to the model learnt', size = 20);